# Multilevel and Marginal Modeling in Python

---

## 1. Introduction
This notebook builds on our earlier introduction to regression modeling with NHANES data. Here, we extend those basic [linear](../02_fitting_models_to_independent_data/09_linear_regression_in_python.ipynb) and [logistic](../02_fitting_models_to_independent_data/10_logistic_regression_in_python.ipynb) regression methods to more advanced approaches for analyzing data with statistical dependencies.

Some of the models in this notebook may take a few minutes to run.

Let's start by importing the necessary libraries.

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

Data often has a multilevel or dependent structure for many reasons. In fact, most real-world datasets show some form of dependence, so independent data should be seen as the exception. In this notebook, we’ll revisit the NHANES data with a focus on dependence caused by clustering, which we’ll define shortly.

We start by reading the data from a CSV file into a Pandas dataframe and remove rows with missing values in the variables we need. (While there are better ways to handle missing data, we use this simple approach for now.)

In addition to the demographic and health variables used previously, we keep two extra variables—SDMVSTRA and SDMVPSU—that will help us define the clustering structure in the data.

In [3]:
# Read the data file
nhanes_df = pd.read_csv("../02_fitting_models_to_independent_data/data/nhanes_2015_2016.csv")

# Drop unused columns, drop rows with any missing values
cols = ["BPXSY1", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "DMDEDUC2", "BMXBMI", "SMQ020", "SDMVSTRA", "SDMVPSU"]
nhanes_df = nhanes_df[cols].dropna()

# Rename columns for convenience
nhanes_df = nhanes_df.rename(columns={
    "BPXSY1": "sbp", 
    "RIDAGEYR": "age", 
    "RIAGENDR": "gender", 
    "RIDRETH1": "ethnicity", 
    "DMDEDUC2": "education",
    "BMXBMI": "bmi", 
    "SMQ020": "smoked", 
    "SDMVSTRA": "stratum", 
    "SDMVPSU": "psu",
})

# Create the labeled versions of `gender` and `education` predictors
nhanes_df["gender_labeled"] = nhanes_df["gender"].replace({1: "Male", 2:"Female"})
nhanes_df["education_labeled"] = nhanes_df["education"].replace({
    1: "lt9",
    2: "xp_11",
    3: "HS",
    4: "SomeCollege",
    5: "College",
    7: np.nan, 
    9: np.nan
})

# Simplify the answers for the `smoked` predictor
nhanes_df["smoked"] = nhanes_df["smoked"].replace({2: 0, 7: np.nan, 9: np.nan})

# Display the first few rows
nhanes_df.head()

,sbp,age,gender,ethnicity,education,bmi,smoked,stratum,psu,gender_labeled,education_labeled
0,128.0,62,1,3,5.0,27.8,1.0,125,1,Male,College
1,146.0,53,1,3,3.0,30.8,1.0,125,1,Male,HS
2,138.0,78,1,3,3.0,28.8,1.0,131,1,Male,HS
3,132.0,56,2,3,5.0,42.4,0.0,131,1,Female,College
4,100.0,42,2,4,4.0,20.3,0.0,126,2,Female,SomeCollege



### Introduction to Clustered Data

Data are often dependent because they are collected using "cluster sampling." In this approach, the population is divided into groups (clusters), a few clusters are selected, and then individuals are sampled from those clusters. This is how NHANES collects its data. Instead of sampling people from all over the country, NHANES sets up examination centers in selected communities and examines many people at each center.

Cluster sampling isn’t the only reason for dependence in data. For example, in longitudinal studies, the same people are measured multiple times, so their measurements are naturally correlated. However, since NHANES uses cluster sampling and isn’t longitudinal, we’ll focus on clustering as our example of dependent data.

In any cluster sample, people within the same cluster tend to be more similar to each other than to people in other clusters. For NHANES, clusters are based on geography, so people from the same community may share similar characteristics. It’s important to account for this dependence in our analysis.

### Clustering Structure in NHANES

The NHANES sampling process is complex, but we’ll keep it simple here. (More details are available [here](https://wwwn.cdc.gov/nchs/nhanes/analyticguidelines.aspx), but you don’t need them for this course.) In short, NHANES selects a sample of US counties, then subregions within those counties, and finally people within those subregions. Because people from the same county live near each other, they are likely to be more similar than people from different counties.

For privacy reasons, NHANES does not provide the actual county identifiers. Instead, we get "masked variance units" (MVUs), which are artificial groups created by combining subregions from different counties. These MVUs aren’t real geographic clusters, but they are designed to mimic them while protecting participants’ privacy.

In this notebook, we’ll treat MVUs as our clusters and examine how clustering affects some NHANES variables.

We can identify each MVU by combining the `stratum` and `psu` variables:

In [4]:
nhanes_df["group"] = 10 * nhanes_df["stratum"] + nhanes_df["psu"]
nhanes_df.head()

,sbp,age,gender,ethnicity,education,bmi,smoked,stratum,psu,gender_labeled,education_labeled,group
0,128.0,62,1,3,5.0,27.8,1.0,125,1,Male,College,1251
1,146.0,53,1,3,3.0,30.8,1.0,125,1,Male,HS,1251
2,138.0,78,1,3,3.0,28.8,1.0,131,1,Male,HS,1311
3,132.0,56,2,3,5.0,42.4,0.0,131,1,Female,College,1311
4,100.0,42,2,4,4.0,20.3,0.0,126,2,Female,SomeCollege,1262



---

## 2. Data Understanding

### Intraclass Correlation
The **intraclass correlation coefficient (ICC)** measures how similar observations are within the same cluster. Unlike Pearson's correlation, the ICC ranges from 0 (no similarity within clusters) to 1 (everyone in a cluster is identical).

We can estimate the ICC using two methods: *marginal regression* and *multilevel regression*. We'll start with **Generalized Estimating Equations (GEE)** to fit marginal models and estimate the ICC for NHANES clusters.

Let's first look at the ICC for systolic bloop pressure `sbp`:

In [5]:
model = sm.GEE.from_formula("sbp ~ 1", groups="group", cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
result = model.fit()
print(result.cov_struct.summary())

The correlation between two observations in the same cluster is 0.030


The estimated ICC is 0.03. This is small, but not negligible. Although the ICC is a kind of correlation, its values aren't directly comparable to Pearson correlation coefficitents. For example, 0.03 would be very small for Person's $r$, but is not unusually small for an ICC.

To better understand clustering in the data, we also calculate ICC values for several other variables used in our analyses.

In [6]:
for col in ["sbp", "age", "bmi", "smoked", "stratum"]:
    model = sm.GEE.from_formula(f"{col} ~ 1", groups="group", cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
    result = model.fit()
    print(f"{col}:", result.cov_struct.summary())

sbp: The correlation between two observations in the same cluster is 0.030
age: The correlation between two observations in the same cluster is 0.035
bmi: The correlation between two observations in the same cluster is 0.039
smoked: The correlation between two observations in the same cluster is 0.026
stratum: The correlation between two observations in the same cluster is 0.959


The values are generally similar to what we saw for blood pressure, except for `stratum`, which is one component of the cluster definition itself, and therefore has a very high ICC.

To illustrate that the ICC values shown above are not consistent with a complete absence of dependence, we simulate 10 sets of random data and calculate the ICC value for each set:

In [7]:
# Set a random seed for reproducibility
np.random.seed(42)

for k in range(10):
    nhanes_df["noise"] = np.random.normal(size=nhanes_df.shape[0])
    model = sm.GEE.from_formula("noise ~ 1", groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
    result = model.fit()
    print(result.cov_struct.summary())

The correlation between two observations in the same cluster is -0.001
The correlation between two observations in the same cluster is -0.002
The correlation between two observations in the same cluster is -0.002
The correlation between two observations in the same cluster is 0.001
The correlation between two observations in the same cluster is -0.000
The correlation between two observations in the same cluster is -0.002
The correlation between two observations in the same cluster is -0.001
The correlation between two observations in the same cluster is 0.000
The correlation between two observations in the same cluster is 0.003
The correlation between two observations in the same cluster is -0.000


We see that the estimated ICC for pure simulated noise is random but highly concentrated near zero, varying from around `-0.002` to `+0.003`.  These values are at least a factor of 10 smaller than the ICC values obtaine with the actual NHANES data.  Thus, while the ICC values for the NHANES data are numerically small, they are much larger than what we would expect to obtain if the observations were independent.

### Conditional Intraclass Correlation
The ICCs we looked at before were marginal -they measured how similar values (like SBP) are within clusters compared to between clusters. However, these "cluster effects" might just reflect demographic differences among clusters. For exampla, clusters with older people may have higher SBP overall.

If we control for age, the ICC might get smaller, since we've accounted for a major source of variation. We'll see this effect in the next analysis.

In [8]:
model = sm.GEE.from_formula("sbp ~ age", groups="group", cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
result = model.fit()
print(result.cov_struct.summary())

The correlation between two observations in the same cluster is 0.019


The ICC for SBP drops from 0.03 to 0.02.  We can now assess whether it
drops even further when we add additional covariates that we know to
be predictive of blood pressure.

In [9]:
model = sm.GEE.from_formula("sbp ~ age + gender_labeled + bmi + C(ethnicity)",
           groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
result = model.fit()
print(result.cov_struct.summary())

The correlation between two observations in the same cluster is 0.013


We see here that the ICC has further reduced, to 0.013, due to controlling for these additional factors including ethnicity.

> **Note:** The variable `ethnicity` is a categorical variable containing 5 levels of race/ethnicity information. Since NHANES categorical variables are coded numerically, Statsmodels would have no way of knowing that these are codes and not quantitative data, thus we must use the C() syntax in the formula above to force this variable to be treated as being categorical. 

---

## 3. Fitting Marginal Linear Models
So far, we've looked at how clustering created dependence in the data. This helps us understand more than just average trends.

When working with dependent data, we can still estimate the means structure (regression coefficients) without considering dependence. However, if we ignore dependence, our standard errors and measures of uncertainty will be incorrect.

To show this, we'll fit two models with the same predictors to the NHANES data. The first uses ordinary least squares (OLS), which assumes independence. The second uses GEE, which accounts for dependence.

In [10]:
# Fit OLS
model1 = sm.OLS.from_formula("sbp ~ age + gender_labeled + bmi + C(ethnicity)", data=nhanes_df)
result1 = model1.fit()

# Fit a marginal GLM 
model2 = sm.GEE.from_formula("sbp ~ age + gender_labeled + bmi + C(ethnicity)", groups="group", cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
result2 = model2. fit()

# Compare estimated coefficients and standard errors
x = pd.DataFrame({"OLS_params": result1.params, "OLS_SE": result1.bse,
                  "GEE_params": result2.params, "GEE_SE": result2.bse})
x

,OLS_params,OLS_SE,GEE_params,GEE_SE
Intercept,91.736583,1.339378,92.168530,1.384309
gender_labeled[T.Male],3.671294,0.453763,3.650245,0.454498
C(ethnicity)[T.2],0.855488,0.819486,0.159296,0.767025
C(ethnicity)[T.3],-1.796132,0.671954,-2.233280,0.760228
C(ethnicity)[T.4],3.813314,0.732355,3.105654,0.881580
C(ethnicity)[T.5],-0.455347,0.808948,-0.439831,0.813675
age,0.478699,0.012901,0.474101,0.018493
bmi,0.278015,0.033285,0.280205,0.038553


In the results above, the estimated coefficients are similar for both OLS and GEE models, but the standard errors are larger with GEE—often by 20-40% for variables like BMI and age. This happens because GEE accounts for clustering and dependence in the data, while OLS assumes independence. As a result, OLS standard errors are not valid when there is clustering, but GEE provides correct estimates as long as the dependence is only within clusters.

---

## 4. Fitting Marginal Logistic Regression
Previously, we used GEE to fit marginal linear models when data are dependent. GEE can also be used for any **generalized linear model (GLM)** with clustered data.

Here, we model smoking history using several demographic predictors. First, we fit a standard GLM (which ignores clustering), then we fit the same model using GEE. Both methods estimate the average relationship (marginal mean structure), but only GEE provides correct standard errors. If you use GLM with clustered data, the standard errors (and anything based on them, like confidence intervals and hypothesis tests) will be incorrect.

In [11]:
# Fit a basic GLM
model1 = sm.GLM.from_formula("smoked ~ age + gender_labeled + education_labeled", family=sm.families.Binomial(), data=nhanes_df)
result1 = model1.fit()

# Fit a marginal GLM using GEE
model2 = sm.GEE.from_formula("smoked ~ age + gender_labeled + education_labeled", groups="group", family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
result2 = model2.fit()

# Compare estimated coefficients and standard errors
x = pd.DataFrame({"OLS_params": result1.params, "OLS_SE": result1.bse, "GEE_params": result2.params, "GEE_SE": result2.bse})
x

,OLS_params,OLS_SE,GEE_params,GEE_SE
Intercept,-2.305999,0.114308,-2.249820,0.140567
gender_labeled[T.Male],0.909597,0.060167,0.908682,0.062342
education_labeled[T.HS],0.943364,0.089663,0.887965,0.095397
education_labeled[T.SomeCollege],0.832227,0.084361,0.771636,0.104449
education_labeled[T.lt9],0.266228,0.109183,0.321784,0.141327
education_labeled[T.xp_11],1.098561,0.106697,1.062149,0.138401
age,0.018257,0.001725,0.017416,0.001803


As expected, the GLM and GEE give very similar estimates for the regression coefficients. However, the standard errors from GEE are larger than those from GLM. This shows that GLM underestimates uncertainty because it ignores data dependence, while GEE gives more accurate standard errors.

When GLM and GEE parameter estimates differ, it's because GEE uses the dependence structure to produce more efficient (accurate) estimates. In summary, compared to GLM, GEE:

* Provides insight into the data's dependent structure.
* Uses this structure to give more accurate standard errors.
* Can produce more accurate parameter estimates.

GLM doesn't account for dependence or provide meaningful standard errors in clustered data -its standard errors can be much too small. Even weak clustering (ICC of 0.02-0.04) can increase standard errors by 10-40%. Although GEE is generally more efficient, GLM estimates are still valid, just less precise in the presence of clustering.

---

## 5. Fitting Multilevel Models

Multilevel modeling is a broad topic, but here we'll focus on using it as another way to handle dependence in clustered data. In this way, multilevel models are an alternative to the marginal regression methods discussed earlier.

For linear regression, multilevel and marginal models are quite similar (they differ more in logistic and nonlinear models). Both approaches estimate the same overall effects, but they use different methods and represent these effects differently.

Multilevel models include **random effects**, which are unobserved variables that capture the influence of clusters. Even though we don't observe these random effects directly, we can estimate their impact from the data, as long as each random effect affects at least two observations.

In our case, we're looking at dependence from a single level of clustering. In multilevel modeling, this means each cluster gets its own random effect -shared by all individuals in that cluster. For example, if one cluster tends to have SBP values about 0.5 units higher, the random effect for that cluster would be 0.5 and would be added to the predicted SBP for everyone in that cluster.

In [12]:
# Fit a multilevel (mixed effects) model to handle dependent data
model = sm.MixedLM.from_formula("sbp ~ age + gender_labeled + bmi + C(ethnicity)", groups="group", data=nhanes_df)
result = model.fit()
print(result.summary())

              Mixed Linear Model Regression Results
Model:                MixedLM   Dependent Variable:   sbp        
No. Observations:     5102      Method:               REML       
No. Groups:           30        Scale:                256.6952   
Min. group size:      106       Log-Likelihood:       -21409.8702
Max. group size:      226       Converged:            Yes        
Mean group size:      170.1                                      
-----------------------------------------------------------------
                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------
Intercept              92.173    1.402 65.752 0.000 89.426 94.921
gender_labeled[T.Male]  3.650    0.452  8.084 0.000  2.765  4.535
C(ethnicity)[T.2]       0.153    0.887  0.172 0.863 -1.586  1.891
C(ethnicity)[T.3]      -2.238    0.758 -2.954 0.003 -3.723 -0.753
C(ethnicity)[T.4]       3.098    0.836  3.707 0.000  1.460  4.737
C(ethnicity)[T.5]      -

The "variance structure parameters" set mixed models apart from marginal models. Here, we have just one: the group-level variance, estimated at 3.615. This means that, on average, random effects between two groups differ by about 2.69 (the square root of `2 * 6.615`). This is a meaningful difference -about the same as the difference between genders or 6 years of aging.

In this example, the mixed model handles dependence in the data, estimates the strength of this dependence, and adjusts both estimates and inference for it -just like GEE. Both methods are widely used and have their own strengths, but neither is always better than the other.

Multilevel models can also estimate ICC values. For a single-level model like this, ICC is the group variance (3.615) divided by the total variance (group variance plus unexplained variance, or 3.615 + 256.7). This gives about 0.014—very similar to the ICC estimated with GEE.

### Predicted Random Effects
Although we can't directly observe the random effects in a multilevel model, we can estimate them from the data. These predicted random effects are called **BLUPs ("Best Linear Unbiased Predictors")**. 

While multilevel modeling usually focuses on the structural parameters, looking at BLUPs can sometimes be helpful. For example, in the NHANES analysis, BLUPs show how unique each county (or MVU) is compared to the overall average.

Below are the predicted random effects for the 30 groups (MVUs) in this analysis:

In [ ]:
blup_list = [v.iloc[0] for v in result.random_effects.values()]
print(blup_list)

[np.float64(-1.6309763436118858), np.float64(-0.08616211402930866), np.float64(-2.042660969749039), np.float64(-0.1474721589703174), np.float64(0.2806232209649395), np.float64(1.5807315390972512), np.float64(0.283347192823595), np.float64(0.13151240310378806), np.float64(-2.0381706331542824), np.float64(0.6176505794890838), np.float64(2.878487707973413), np.float64(-0.5193637318761565), np.float64(2.0649670924026267), np.float64(1.5212808931265185), np.float64(-1.2619747720774532), np.float64(0.9808461060612339), np.float64(0.11803118572912655), np.float64(-0.12839673871518267), np.float64(-0.38486223890136545), np.float64(-3.582111121241415), np.float64(-3.271016524089456), np.float64(-0.8295384666982739), np.float64(-0.8841714000567727), np.float64(2.7906567857627405), np.float64(-0.5852008795225484), np.float64(1.1982910933210809), np.float64(-0.1956917281747792), np.float64(1.9555145010226551), np.float64(-0.30555914261143186), np.float64(1.4913886626026707)]
